# Heterogeneous Graph Visualization

Visualize the bipartite graph structure that separates spacetime from field content.

In [ ]:
import os, sys

IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/qft_graph'
    !pip install -q torch-geometric omegaconf plotly
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
    os.chdir(PROJECT_ROOT)
else:
    sys.path.insert(0, '../src')

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go

from qft_graph.config import LatticeConfig
from qft_graph.lattice.hypercubic import HypercubicLattice
from qft_graph.fields.scalar import ScalarField
from qft_graph.graphs.builder import HeteroGraphBuilder
from qft_graph.graphs.edge_types import ADJACENT, inhabits_edge

plt.style.use('dark_background')
%matplotlib inline

In [ ]:
# Use small lattice for visibility
lattice = HypercubicLattice(LatticeConfig(dimensions=(4, 4)))
scalar = ScalarField()
builder = HeteroGraphBuilder(lattice, [scalar])

torch.manual_seed(42)
phi = scalar.initialize(lattice.num_sites(), mode='hot')
data = builder.build({'scalar': phi})

print('Node types:', data.node_types)
print('Edge types:', data.edge_types)
print(f'Spacetime nodes: {data["spacetime"].num_nodes}')
print(f'Scalar nodes: {data["scalar"].num_nodes}')
print(f'Adjacency edges: {data[ADJACENT].edge_index.shape[1]}')
print(f'Inhabits edges: {data[inhabits_edge("scalar")].edge_index.shape[1]}')

In [ ]:
# --- Interactive 3D Visualization with Plotly ---
# Spacetime nodes on the z=0 plane, scalar field nodes on the z=-2 plane
# Rotate, zoom, and pan to explore the graph structure

coords = data['spacetime'].x[:, :2].numpy()  # x, y coordinates
phi_vals = data['scalar'].x[:, 0].numpy()

# 3D positions: spacetime at z=0, field nodes at z=-2
st_x, st_y, st_z = coords[:, 0], coords[:, 1], np.zeros(len(coords))
f_x, f_y, f_z = coords[:, 0], coords[:, 1], np.full(len(coords), -2.0)

traces = []

# --- Adjacency edges (spacetime ↔ spacetime, blue) ---
adj = data[ADJACENT].edge_index.numpy()
adj_xe, adj_ye, adj_ze = [], [], []
for i in range(adj.shape[1]):
    src, dst = adj[0, i], adj[1, i]
    # Skip wrap-around edges for clarity
    if np.linalg.norm(coords[src] - coords[dst]) < 1.5:
        adj_xe += [st_x[src], st_x[dst], None]
        adj_ye += [st_y[src], st_y[dst], None]
        adj_ze += [st_z[src], st_z[dst], None]

traces.append(go.Scatter3d(
    x=adj_xe, y=adj_ye, z=adj_ze,
    mode='lines', line=dict(color='#4488ff', width=3),
    name='Adjacent (spacetime↔spacetime)',
    hoverinfo='skip'
))

# --- Inhabits edges (field → spacetime, orange dashed) ---
inh_xe, inh_ye, inh_ze = [], [], []
for i in range(len(coords)):
    inh_xe += [f_x[i], st_x[i], None]
    inh_ye += [f_y[i], st_y[i], None]
    inh_ze += [f_z[i], st_z[i], None]

traces.append(go.Scatter3d(
    x=inh_xe, y=inh_ye, z=inh_ze,
    mode='lines', line=dict(color='#ff8844', width=2, dash='dash'),
    name='Inhabits (scalar→spacetime)',
    hoverinfo='skip'
))

# --- Spacetime nodes (blue spheres) ---
traces.append(go.Scatter3d(
    x=st_x, y=st_y, z=st_z,
    mode='markers+text',
    marker=dict(size=8, color='#4488ff', line=dict(color='white', width=1)),
    text=[f'ST{i}' for i in range(len(coords))],
    textposition='top center',
    textfont=dict(size=9, color='white'),
    name='Spacetime nodes',
    hovertemplate='<b>ST%{text}</b><br>x=%{x:.0f}, y=%{y:.0f}<extra></extra>'
))

# --- Scalar field nodes (colored by φ value) ---
# Normalize phi for colorscale
phi_abs_max = max(abs(phi_vals.min()), abs(phi_vals.max()))
traces.append(go.Scatter3d(
    x=f_x, y=f_y, z=f_z,
    mode='markers+text',
    marker=dict(
        size=8,
        color=phi_vals,
        colorscale='RdBu_r',
        cmin=-phi_abs_max, cmax=phi_abs_max,
        colorbar=dict(title='φ value', x=1.02, len=0.5),
        line=dict(color='white', width=1)
    ),
    text=[f'{v:.2f}' for v in phi_vals],
    textposition='bottom center',
    textfont=dict(size=8, color='white'),
    name='Scalar field nodes',
    hovertemplate='<b>φ=%{text}</b><br>site (%{x:.0f}, %{y:.0f})<extra></extra>'
))

# --- Semi-transparent planes to show the two layers ---
plane_x = np.array([[-0.5, 3.5], [-0.5, 3.5]])
plane_y = np.array([[-0.5, -0.5], [3.5, 3.5]])

# Spacetime plane (z=0)
traces.append(go.Surface(
    x=plane_x, y=plane_y, z=np.zeros_like(plane_x),
    colorscale=[[0, 'rgba(68,136,255,0.08)'], [1, 'rgba(68,136,255,0.08)']],
    showscale=False, name='Spacetime plane',
    hoverinfo='skip'
))

# Field plane (z=-2)
traces.append(go.Surface(
    x=plane_x, y=plane_y, z=np.full_like(plane_x, -2.0),
    colorscale=[[0, 'rgba(255,136,68,0.08)'], [1, 'rgba(255,136,68,0.08)']],
    showscale=False, name='Field plane',
    hoverinfo='skip'
))

fig = go.Figure(data=traces)

fig.update_layout(
    title=dict(
        text='Heterogeneous Bipartite Graph (4×4 lattice)<br>'
             '<sub>Blue = spacetime lattice (z=0) | Orange = scalar field (z=−2) | Drag to rotate</sub>',
        font=dict(size=16)
    ),
    scene=dict(
        xaxis=dict(title='x', range=[-0.8, 3.8], backgroundcolor='rgb(20,20,30)'),
        yaxis=dict(title='y', range=[-0.8, 3.8], backgroundcolor='rgb(20,20,30)'),
        zaxis=dict(title='Layer', range=[-3, 1], backgroundcolor='rgb(20,20,30)'),
        aspectmode='manual',
        aspectratio=dict(x=1, y=1, z=0.6),
        camera=dict(eye=dict(x=1.8, y=1.8, z=1.2)),
    ),
    paper_bgcolor='rgb(15,15,25)',
    plot_bgcolor='rgb(15,15,25)',
    font=dict(color='white'),
    legend=dict(x=0, y=1, bgcolor='rgba(0,0,0,0.5)'),
    width=900, height=700,
    margin=dict(l=0, r=0, t=80, b=0),
)

fig.show()